
#### **My observation of this section  :**

##### This data was highly imbalanced. An imbalanced dataset is a classification data scenario where the distribution of classes is not equal, meaning one class (majority) significantly outnumbers another (minority), in imbalanced datasets, models are biased toward the majority class. To address this,


##### I will have to remove non-informative features like identifiers and constant columns, as they do not contribute to predictive power. I will retain features that have behavioral or financial relevance to churn.

In [52]:
import pandas as pd

df = pd.read_csv("../data/raw/European_Bank.csv")
df.shape

(10000, 14)

##### We got 10000 rows, 14 features.

In [53]:
df.head(15)

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,2025,15574012,Chu,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,2025,15592531,Bartlett,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,2025,15656148,Obinna,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1
8,2025,15792365,He,501,France,Male,44,4,142051.07,2,0,1,74940.50,0
9,2025,15592389,H?,684,France,Male,27,2,134603.88,1,1,1,71725.73,0


#### To have a look at the data

In [54]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


##### Checked the datatypes of the columns, the main columns we will focus on will be exited thats an Int type.

In [55]:
df.isnull().sum()

Year               0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

##### There are no missing values in this dataset
##### it will simplify preprocessing, but I still need to check feature quality and relevance.

In [56]:
df.columns

Index(['Year', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender',
       'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

##### These are the columns

In [57]:
df['Exited'].value_counts()  # so we have binary values 1s and 0s. 1 - churn, 2 - stayed. count occurrences

Exited
0    7963
1    2037
Name: count, dtype: int64

##### In Exited columns we have 7963 values of 0s (non churn customers), 2037 of customers who stayed.

In [58]:
df['Exited'].value_counts(normalize=True)

Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64

##### Distribution Proportion of 0 - 79.6% and 1 - 20.4%
##### There is a class imbalance, since one class (0) has more data than the other(1)
##### Class imbalance makes accuracy unreliable and biases the model toward the majority class, which is dangerous in churn prediction where detecting minority cases is critical.
##### In imbalanced datasets, accuracy can be misleading because a model can achieve high accuracy by predicting the majority class, while failing to detect the minority class patterns.

In [59]:
df.describe()

,Year,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.0,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,2025.0,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,0.0,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,2025.0,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2025.0,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,2025.0,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,2025.0,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,2025.0,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


##### Lets only pay attention to the columns that can help predict the churn.
##### And if a column doesn’t logically affect churn will drop it later anyway

In [60]:
df['Geography'].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

##### 50.01% cms are from France
##### 25.09% cms are from Germany
##### 24.77% cms are from Spain
##### I see that the data is not evenly distributed by country, France dominates.

In [61]:
df['Gender'].value_counts()

Gender
Male      5457
Female    4543
Name: count, dtype: int64

##### 54% male users, 45% female.

In [62]:
df.groupby('Geography')['Exited'].mean()

Geography
France     0.161548
Germany    0.324432
Spain      0.166734
Name: Exited, dtype: float64

##### Average (Mean) % of customers who exited based on 'Geography'
##### 16%  France
##### 32% Germany
##### 16% Spain

In [63]:
df.groupby('Gender')['Exited'].mean()

Gender
Female    0.250715
Male      0.164559
Name: Exited, dtype: float64

##### Average (Mean) % of customers who exited based on 'Gender'
##### 25% F
##### 16% M

In [64]:
df.groupby('IsActiveMember')['Exited'].mean()

IsActiveMember
0    0.268509
1    0.142691
Name: Exited, dtype: float64

##### Average (Mean) % of customers who exited based on 'IsActiveMember'
##### 26% Active member Stayed
##### 14% Active Member Exited

In [65]:
df.groupby('NumOfProducts')['Exited'].mean()

NumOfProducts
1    0.277144
2    0.075817
3    0.827068
4    1.000000
Name: Exited, dtype: float64

##### Average (Mean) % of customers who exited based on Number of Products they had.
##### Members with 1 Product - 27% of them Exited
##### Members with 2 Products - 7% of them Exited
##### Members with 3 Products - 82% of them Exited
##### Members with 4 Products - 100% of them Exited

In [66]:
df.groupby('HasCrCard')['Exited'].mean()

HasCrCard
0    0.208149
1    0.201843
Name: Exited, dtype: float64

##### 20 % of al the People who have card left.
##### 20% % of all the People who does not have card left.

In [67]:
df['NumOfProducts'].value_counts()

NumOfProducts
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64

##### Number of Products Distribution using value_counts()

- 5084 customers have 1 product
- 4590 customers have 2 products
- 266 customers have 3 products
- 60 customers have 4 products

Most customers (around 97%) have either 1 or 2 products, while very few customers have 3 or 4 products.

This means that insights related to customers with 3 or 4 products may be less reliable due to the small number of observations.

In [68]:
df.groupby(pd.cut(df['Age'], bins=5))['Exited'].mean()

Age
(17.926, 32.8]    0.076344
(32.8, 47.6]      0.188182
(47.6, 62.4]      0.529978
(62.4, 77.2]      0.214925
(77.2, 92.0]      0.041667
Name: Exited, dtype: float64

##### Percentage of churn as per age groups, we have 5 bins of age and we analyzed that :

##### Young customers (18–32) → low churn
##### Middle-aged (47–62) → highest churn
##### Very old (77+) → low churn again

- Simple interpretation : Customers around 45–60 age group are most likely to leave

In [69]:
df.groupby(pd.cut(df['Balance'], bins=5))['Exited'].mean()

Balance
(-250.898, 50179.618]       0.142470
(50179.618, 100359.236]     0.199609
(100359.236, 150538.854]    0.257837
(150538.854, 200718.472]    0.217486
(200718.472, 250898.09]     0.593750
Name: Exited, dtype: float64


##### Percentage of churn as per the balance (Outstanding Balance)
##### 0–50k - 14%
##### 50k–100k - 20%
##### 100k–150k - 26%
##### 150k–200k - 21%
##### 200k+  - 59%
- Simple interpretation : Low balance → low risk, Medium balance → moderate risk, Very high balance → high risk.


---
---
###  EDA Summary

- The dataset is clean with no missing values and has a slight class imbalance (80% non-churn, 20% churn)
- Customer activity, number of products, age, balance, and geography are key factors influencing churn
- Inactive customers and those with only one product are more likely to churn
- Customers aged 45–60 show the highest churn rates
- Customers with higher balances tend to churn more
- Germany has a significantly higher churn rate compared to other regions
- Credit card ownership has minimal impact on churn
- Overall, customer engagement and product usage are the strongest indicators of churn

#### **As identified major churn drivers:**

##### Strong signals
- IsActiveMember
- NumOfProducts
- Age (middle-aged risk)
- Balance (high balance risk)
- Geography (Germany)
---
---

## SUMMARY : Exploratory Data Analysis

The dataset was first examined to understand its structure, quality, and underlying patterns influencing customer churn. The dataset consists of 10,000 customer records with no missing values, indicating a clean and reliable data source for analysis.

An initial assessment of the target variable revealed a slight class imbalance, with approximately 79.6% of customers retained and 20.4% having churned. While not extreme, this imbalance highlights the need for careful evaluation metrics beyond accuracy during model development.

Several key patterns were identified during the analysis:

Geography emerged as a significant factor, with customers in Germany exhibiting nearly double the churn rate compared to those in France and Spain.
Customer activity showed a strong relationship with churn, where inactive customers were significantly more likely to leave than active ones.
Number of products played a critical role, with customers holding two products showing the lowest churn rates, while those with only one product or more than two products demonstrated higher churn tendencies.
Age revealed a non-linear relationship with churn, with middle-aged customers (approximately 45–60 years) exhibiting the highest churn rates.
Account balance indicated that customers with higher balances tend to churn more, suggesting potential dissatisfaction among high-value customers.
Credit card ownership showed minimal impact on churn, indicating it is not a strong predictive feature.
It is important to note that certain observations, particularly for customers with three or more products and those with very high balances, are based on smaller sample sizes and should be interpreted with caution.

Overall, the analysis highlights that customer engagement and product usage are the most influential factors affecting churn, providing a strong foundation for building predictive models and designing targeted retention strategies.

#### **Next Step Data Preprocessing!**